In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import h5py

from astropy.table import Table
import glob

In [ ]:
snapshot_list = [i for i in range(87, 99)]  # Your list of snapshot numbers

dfs = []
with h5py.File('stellar_circs.hdf5', 'r') as f:
    for snapshot_n in snapshot_list:
        data = np.array(f[f'Snapshot_{snapshot_n}']['MassTensorEigenVals'])
        names = np.array(f[f'Snapshot_{snapshot_n}']['SubfindID'])
        
        df = pd.DataFrame({
            'snapshot_n': snapshot_n,
            'SubfindID': names,
            'c_axis': data[:, 0],
            'b_axis': data[:, 1],
            'a_axis': data[:, 2]
        })
        dfs.append(df)

result_df = pd.concat(dfs, ignore_index=True)
result_df

In [ ]:
result_df['Name'] = result_df.apply(lambda x: f'{x.snapshot_n:.0f}_{x.SubfindID:.0f}', axis=1)

In [ ]:
mangia_table = Table.read('MaNGIA_catalog.fits', format='fits').to_pandas()

In [ ]:
mangia_table['Name'] = mangia_table.apply(lambda x: f'{x.snapshot:.0f}_{x.subhalo_id:.0f}', axis=1)
mangia_table['Name_ext'] = mangia_table.apply(lambda x: f'{x.snapshot:.0f}_{x.subhalo_id:.0f}_{x['view']}', axis=1)

In [ ]:
joined = pd.merge(result_df, mangia_table, on='Name', how='inner', suffixes=('', '_mangia'))

In [ ]:
joined

In [ ]:
dir_name = 'VELOCITY_VDISP_FLUX_MAPS'
files_list = glob.glob(f'{dir_name}/TNG50*.fits')

In [ ]:
files_list = [x.split('/')[-1] for x in files_list]

In [ ]:
ids = [x.split('-')[2] for x in files_list]
n_snap = [x.split('-')[1] for x in files_list]
n_view = [x.split('-')[3] for x in files_list]

In [ ]:
names = [x + '_' + y for x, y in zip(n_snap, ids)]
names_ex = [x + '_' + y + '_' + z for x, y, z in zip(n_snap, ids, n_view)]

In [ ]:
ifu_files = pd.DataFrame({'filename': files_list, 'sub_id': ids, 'snapshot': n_snap, 'view': n_view, 'Name': names, 'Name_ext': names_ex})

In [ ]:
joined = pd.merge(ifu_files, joined, on='Name_ext', how='left', suffixes=('', '_joined'))

In [ ]:
joined.to_csv('files_list_and_axis.csv', index=False)

In [ ]:
joined = pd.read_csv('files_list_and_axis.csv')

In [ ]:
def ellips_type(a, b, c):
    if b/a > 0.9:
        if c/a > 0.9:
            return 0 # sphere
        elif c/a < 0.85:
            return 1 # oblate
        else:
            return -1
    elif b/a < 0.85:
        if c/b > 0.9:
            return 2 # prolate
        elif c/b < 0.85:
            return 3 # tri-axial
        else:
            return -1
    else:
        return -1

In [ ]:
joined['type'] = joined.apply(lambda x: ellips_type(x['a_axis'], x['b_axis'], x['c_axis']), axis=1)

In [ ]:
joined['c/a'] = joined['c_axis']/joined['a_axis']
joined['b/a'] = joined['b_axis']/joined['a_axis']

In [ ]:
plt.scatter(joined['b/a'], joined['c/a'], color='k')
mask_sphere = (joined['type'] == 0)
mask_prolate = (joined['type'] == 2)
mask_oblate = (joined['type'] == 1)
mask_triaxial = (joined['type'] == 3)
plt.scatter(joined[mask_sphere]['b/a'], joined[mask_sphere]['c/a'], color='r')
plt.scatter(joined[mask_prolate]['b/a'], joined[mask_prolate]['c/a'], color='g')
plt.scatter(joined[mask_oblate]['b/a'], joined[mask_oblate]['c/a'], color='b')
plt.scatter(joined[mask_triaxial]['b/a'], joined[mask_triaxial]['c/a'], color='y')

In [41]:
joined.to_csv('files_list_and_axis.csv', index=False)